# TensorFlow 실습 002 — MLP 차원 이해 + 학습 비교 실험

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/glasslego/2026-ml-study/blob/main/notebooks/tensorflow_002_mlp_shape_and_training_experiments.ipynb)

`tensorflow_001.ipynb`의 다음 단계 실습입니다.
이번 노트북은 **MLP의 차원(shape) 해석**을 먼저 분명히 한 뒤,
같은 데이터셋으로 **Batch Normalization**, **Optimizer**, **Adam learning rate**, **batch size**를 바꿨을 때
학습 곡선과 최종 성능이 어떻게 달라지는지 비교합니다.

## 이 노트북에서 꼭 잡고 갈 것

1. `X.shape = (샘플 수, 피처 수)`를 정확히 읽는 방법
2. 미니배치가 들어가면 텐서 shape이 어떻게 바뀌는지
3. `input_dim`, `hidden_units`, `output_dim`을 어떻게 정하는지
4. BatchNorm은 **shape을 바꾸지 않고 분포를 안정화**한다는 점
5. Optimizer와 learning rate는 **업데이트 방식**을 바꾸지만 출력 차원은 바꾸지 않는다는 점

## 데이터셋

- 기존 실습과 같은 **Wine 분류 데이터셋**을 사용합니다.
- 샘플 수가 작아서 실험이 빠르게 끝납니다.
- 다만 데이터가 작기 때문에 어떤 실험에서는 차이가 아주 극적이지 않을 수 있습니다.
  그 대신 **곡선의 안정성**, **수렴 속도**, **과적합 정도**를 보는 연습에 적합합니다.


---
## Cell 1. 라이브러리 준비

Colab에서는 보통 TensorFlow가 이미 설치되어 있지만,
혹시 빠져 있으면 이 셀에서 자동 설치하도록 구성합니다.


In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    'tensorflow': 'tensorflow',
    'numpy': 'numpy',
    'pandas': 'pandas',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'sklearn': 'scikit-learn',
}

missing_packages = [
    pip_name
    for module_name, pip_name in required_packages.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    print('설치가 필요한 패키지:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('필요한 패키지가 이미 설치되어 있습니다.')

import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Activation, BatchNormalization, Dense, Input
from tensorflow.keras.optimizers import Adam, RMSprop, SGD
from tensorflow.keras.utils import to_categorical

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.unicode_minus'] = False

print('TensorFlow version:', tf.__version__)
print('NumPy version:', np.__version__)
print('Pandas version:', pd.__version__)


---
## Cell 2. 랜덤 시드 고정

같은 실험 조건이면 결과가 최대한 비슷하게 나오도록 시드를 고정합니다.
하이퍼파라미터만 바꾸고 다른 조건은 고정해야 비교가 의미 있습니다.


In [ ]:
SEED = 42

np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
    print('TensorFlow deterministic ops 활성화 완료')
except Exception as error:
    print('deterministic ops 설정은 환경에 따라 생략될 수 있습니다:', error)

print('고정한 시드:', SEED)


---
## Cell 3. 원본 데이터 shape 읽기

여기서 가장 중요한 포인트는 다음입니다.

- `X.shape = (샘플 수, 피처 수)`
- `y.shape = (샘플 수,)` 또는 원-핫 인코딩 후 `(샘플 수, 클래스 수)`

Wine 데이터셋은 샘플 178개, 피처 13개, 클래스 3개입니다.
즉, **178행 13열 표**라고 생각하면 됩니다.

- 행(axis=0): 샘플 수
- 열(axis=1): 피처 수


In [ ]:
wine = load_wine()

X_raw = wine.data.astype('float32')
y_raw = wine.target.astype('int32')
feature_names = wine.feature_names
class_names = wine.target_names

print('X_raw.shape =', X_raw.shape)
print('y_raw.shape =', y_raw.shape)
print('클래스 이름 =', class_names)
print()

print('샘플 1개(X_raw[0])의 shape =', X_raw[0].shape)
print('샘플 5개(X_raw[:5])의 shape =', X_raw[:5].shape)
print()

print('첫 번째 샘플의 피처 값:')
print(pd.Series(X_raw[0], index=feature_names))
print()

print('첫 번째 샘플의 정답 레이블 y_raw[0] =', y_raw[0])
print('첫 5개 정답 레이블 y_raw[:5] =', y_raw[:5])


---
## Cell 4. 데이터 분할 + 스케일링 + 원-핫 인코딩

이번 실습에서는 데이터를 아래처럼 나눕니다.

1. 전체 데이터 → train/test
2. train 일부를 다시 validation으로 분리

최종적으로 세 집합을 사용합니다.

- train: 실제 학습에 사용
- validation: epoch 동안 모델 선택/비교에 사용
- test: 마지막 최종 성능 확인

그리고 입력 피처는 `StandardScaler`로 표준화합니다.
**입력 표준화**와 **BatchNorm**은 역할이 다르므로 둘 다 함께 보는 것이 좋습니다.


In [ ]:
X_train_full, X_test, y_train_full_raw, y_test_raw = train_test_split(
    X_raw,
    y_raw,
    test_size=0.2,
    random_state=SEED,
    stratify=y_raw,
)

X_train, X_val, y_train_raw, y_val_raw = train_test_split(
    X_train_full,
    y_train_full_raw,
    test_size=0.25,
    random_state=SEED,
    stratify=y_train_full_raw,
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype('float32')
X_val = scaler.transform(X_val).astype('float32')
X_test = scaler.transform(X_test).astype('float32')

y_train = to_categorical(y_train_raw)
y_val = to_categorical(y_val_raw)
y_test = to_categorical(y_test_raw)

print('학습 입력 X_train.shape =', X_train.shape)
print('검증 입력 X_val.shape   =', X_val.shape)
print('테스트 입력 X_test.shape =', X_test.shape)
print()
print('학습 정답 y_train_raw.shape =', y_train_raw.shape)
print('원-핫 인코딩 후 y_train.shape =', y_train.shape)
print('원-핫 인코딩 후 y_val.shape   =', y_val.shape)
print('원-핫 인코딩 후 y_test.shape  =', y_test.shape)
print()

split_summary = pd.DataFrame(
    {
        'split': ['train', 'validation', 'test'],
        'sample_count': [len(X_train), len(X_val), len(X_test)],
        'feature_count': [X_train.shape[1], X_val.shape[1], X_test.shape[1]],
        'label_shape': [str(y_train.shape), str(y_val.shape), str(y_test.shape)],
    }
)
display(split_summary)


---
## Cell 5. 차원 해석을 표로 정리

여기서 많이 헷갈리는 포인트를 표로 정리합니다.

- `X_train.shape = (106, 13)`이면
  - 샘플 106개
  - 각 샘플마다 피처 13개
- `y_train.shape = (106, 3)`이면
  - 샘플 106개
  - 각 샘플의 정답을 3개 클래스 확률 벡터로 표현

즉, **axis=0은 항상 샘플 축**이라고 생각하면 이해가 쉬워집니다.


In [ ]:
batch_example_size = 8

dimension_table = pd.DataFrame(
    [
        ['원본 입력 전체', str(X_raw.shape), '샘플 축=178, 피처 축=13'],
        ['원본 정답 전체', str(y_raw.shape), '샘플 축=178, 레이블은 정수 1개'],
        ['학습 입력 전체', str(X_train.shape), '샘플 축=106, 피처 축=13'],
        ['학습 정답 전체', str(y_train.shape), '샘플 축=106, 클래스 축=3'],
        ['샘플 1개', str(X_train[0].shape), '피처만 남아서 (13,)'],
        [f'미니배치 {batch_example_size}개', str(X_train[:batch_example_size].shape), f'배치 축={batch_example_size}, 피처 축=13'],
        [f'미니배치 정답 {batch_example_size}개', str(y_train[:batch_example_size].shape), f'배치 축={batch_example_size}, 클래스 축=3'],
    ],
    columns=['대상', 'shape', '해석'],
)
display(dimension_table)


---
## Cell 6. Hidden layer / Output layer 차원은 어떻게 정하나?

이번 실습에서는 아래 구조를 예제로 사용합니다.

```
입력: (batch_size, 13)
  ↓
Dense(32)
  ↓
BatchNorm
  ↓
ReLU
  ↓
Dense(16)
  ↓
BatchNorm
  ↓
ReLU
  ↓
Dense(3) + Softmax
```

### 차원 해석

- `input_dim = 13`
  - 데이터의 피처 개수와 같아야 합니다.
- `hidden_units = [32, 16]`
  - 설계자가 정하는 하이퍼파라미터입니다.
  - 첫 은닉층 출력: `(batch_size, 32)`
  - 두 번째 은닉층 출력: `(batch_size, 16)`
- `output_dim = 3`
  - 클래스 개수와 같아야 합니다.
  - 다중분류에서는 마지막 출력 shape이 `(batch_size, 클래스 수)`가 됩니다.


In [ ]:
INPUT_DIM = X_train.shape[1]
NUM_CLASSES = y_train.shape[1]
PROBE_HIDDEN_UNITS = (32, 16)

probe_model = Sequential(name='shape_probe_model')
probe_model.add(Input(shape=(INPUT_DIM,)))
probe_model.add(Dense(PROBE_HIDDEN_UNITS[0]))
probe_model.add(BatchNormalization())
probe_model.add(Activation('relu'))
probe_model.add(Dense(PROBE_HIDDEN_UNITS[1]))
probe_model.add(BatchNormalization())
probe_model.add(Activation('relu'))
probe_model.add(Dense(NUM_CLASSES, activation='softmax'))

probe_model.summary()


---
## Cell 7. 레이어를 통과할 때 shape이 어떻게 변하는지 직접 보기

`batch_size = 4`인 미니배치를 모델에 통과시키면서 shape이 어떻게 바뀌는지 확인합니다.

핵심 포인트:

- Dense는 **마지막 축(feature 축)** 의 크기를 바꿉니다.
- BatchNorm은 **shape은 유지**하고 값의 분포만 조정합니다.
- Activation도 **shape은 유지**합니다.
- 마지막 Dense + Softmax 후에는 `(batch_size, 클래스 수)`가 됩니다.


In [ ]:
probe_batch = tf.convert_to_tensor(X_train[:4], dtype=tf.float32)
current_tensor = probe_batch

print('입력 미니배치 shape =', current_tensor.shape)

shape_trace_rows = [
    ['input_batch', str(tuple(current_tensor.shape)), '배치 4개, 피처 13개'],
]

for layer in probe_model.layers:
    current_tensor = layer(current_tensor, training=False)
    shape_trace_rows.append(
        [layer.name, str(tuple(current_tensor.shape)), layer.__class__.__name__]
    )
    print(f'{layer.name:20s} -> {current_tensor.shape}')

shape_trace = pd.DataFrame(shape_trace_rows, columns=['단계', 'shape', '설명'])
display(shape_trace)


---
## Cell 8. 파라미터 수를 수식으로 확인하기

Dense 레이어의 파라미터 수는 아래 공식으로 계산합니다.

```
가중치 수 = 입력 차원 × 출력 차원
편향 수   = 출력 차원
전체 수   = (입력 차원 × 출력 차원) + 출력 차원
```

예를 들어 첫 Dense(32)는

- 입력 차원 13
- 출력 차원 32
- 가중치 수 = 13 × 32 = 416
- 편향 수 = 32
- 전체 = 448

BatchNorm은 출력 차원을 바꾸지 않지만,
`gamma`, `beta`, `moving_mean`, `moving_variance` 같은 파라미터/상태를 가집니다.


In [ ]:
dense_1_weight = INPUT_DIM * PROBE_HIDDEN_UNITS[0]
dense_1_bias = PROBE_HIDDEN_UNITS[0]

dense_2_weight = PROBE_HIDDEN_UNITS[0] * PROBE_HIDDEN_UNITS[1]
dense_2_bias = PROBE_HIDDEN_UNITS[1]

output_weight = PROBE_HIDDEN_UNITS[1] * NUM_CLASSES
output_bias = NUM_CLASSES

param_table = pd.DataFrame(
    [
        ['Dense(32)', INPUT_DIM, PROBE_HIDDEN_UNITS[0], dense_1_weight, dense_1_bias, dense_1_weight + dense_1_bias],
        ['BatchNorm(32)', PROBE_HIDDEN_UNITS[0], PROBE_HIDDEN_UNITS[0], 32, 32, 128],
        ['Dense(16)', PROBE_HIDDEN_UNITS[0], PROBE_HIDDEN_UNITS[1], dense_2_weight, dense_2_bias, dense_2_weight + dense_2_bias],
        ['BatchNorm(16)', PROBE_HIDDEN_UNITS[1], PROBE_HIDDEN_UNITS[1], 16, 16, 64],
        ['Dense(3)', PROBE_HIDDEN_UNITS[1], NUM_CLASSES, output_weight, output_bias, output_weight + output_bias],
    ],
    columns=['레이어', '입력 차원', '출력 차원', '주요 가중치 수', 'bias/추가 수', '총 파라미터 수'],
)
display(param_table)


---
## Cell 9. 공통 실험 함수 정의

이제부터는 같은 데이터와 같은 모델 뼈대를 두고,
아래 요소만 바꾸면서 학습을 비교합니다.

- BatchNorm 사용 여부
- Optimizer 종류
- Adam learning rate
- Batch size

**중요:**
이 요소들은 학습 방식이나 수렴 속도를 바꾸지만,
`입력 차원`, `은닉층 차원`, `출력 차원` 자체를 바꾸지는 않습니다.


In [ ]:
DEFAULT_HIDDEN_UNITS = (64, 32)
DEFAULT_EPOCHS = 40
DEFAULT_BATCH_SIZE = 16


def reset_all_seeds(seed: int = SEED) -> None:
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)


def build_mlp(
    input_dim: int,
    num_classes: int,
    hidden_units: tuple[int, ...] = DEFAULT_HIDDEN_UNITS,
    activation: str = 'relu',
    use_batch_norm: bool = False,
) -> Sequential:
    model = Sequential(name='wine_mlp')
    model.add(Input(shape=(input_dim,)))

    for units in hidden_units:
        model.add(Dense(units))
        if use_batch_norm:
            model.add(BatchNormalization())
        model.add(Activation(activation))

    model.add(Dense(num_classes, activation='softmax'))
    return model


def make_optimizer(
    optimizer_name: str,
    learning_rate: float,
    momentum: float = 0.0,
    beta_1: float = 0.9,
    beta_2: float = 0.999,
):
    name = optimizer_name.lower()

    if name == 'sgd':
        return SGD(learning_rate=learning_rate, momentum=momentum)
    if name == 'rmsprop':
        return RMSprop(learning_rate=learning_rate)
    if name == 'adam':
        return Adam(learning_rate=learning_rate, beta_1=beta_1, beta_2=beta_2)

    raise ValueError(f'지원하지 않는 optimizer입니다: {optimizer_name}')


def train_single_experiment(config: dict) -> tuple[Sequential, dict, dict]:
    tf.keras.backend.clear_session()
    reset_all_seeds(config.get('seed', SEED))

    model = build_mlp(
        input_dim=INPUT_DIM,
        num_classes=NUM_CLASSES,
        hidden_units=config.get('hidden_units', DEFAULT_HIDDEN_UNITS),
        activation=config.get('activation', 'relu'),
        use_batch_norm=config.get('use_batch_norm', False),
    )

    optimizer = make_optimizer(
        optimizer_name=config.get('optimizer_name', 'adam'),
        learning_rate=config.get('learning_rate', 1e-3),
        momentum=config.get('momentum', 0.0),
        beta_1=config.get('beta_1', 0.9),
        beta_2=config.get('beta_2', 0.999),
    )

    model.compile(
        loss='categorical_crossentropy',
        optimizer=optimizer,
        metrics=['accuracy'],
    )

    start_time = time.perf_counter()
    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=config.get('epochs', DEFAULT_EPOCHS),
        batch_size=config.get('batch_size', DEFAULT_BATCH_SIZE),
        verbose=0,
        shuffle=True,
    )
    elapsed = time.perf_counter() - start_time

    train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
    val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

    result = {
        'name': config['name'],
        'use_batch_norm': config.get('use_batch_norm', False),
        'optimizer_name': config.get('optimizer_name', 'adam'),
        'learning_rate': config.get('learning_rate', np.nan),
        'momentum': config.get('momentum', np.nan),
        'beta_1': config.get('beta_1', np.nan),
        'beta_2': config.get('beta_2', np.nan),
        'batch_size': config.get('batch_size', DEFAULT_BATCH_SIZE),
        'hidden_units': str(config.get('hidden_units', DEFAULT_HIDDEN_UNITS)),
        'train_acc': float(train_acc),
        'val_acc': float(val_acc),
        'test_acc': float(test_acc),
        'train_loss': float(train_loss),
        'val_loss': float(val_loss),
        'test_loss': float(test_loss),
        'best_val_acc': float(np.max(history.history['val_accuracy'])),
        'best_epoch': int(np.argmax(history.history['val_accuracy']) + 1),
        'seconds': round(elapsed, 3),
    }

    return model, history.history, result


def run_experiments(configs: list[dict]) -> tuple[dict, pd.DataFrame]:
    histories = {}
    results = []

    for config in configs:
        _, history, result = train_single_experiment(config)
        histories[config['name']] = history
        results.append(result)
        print(f"완료: {config['name']}")

    result_df = pd.DataFrame(results)
    result_df['generalization_gap'] = result_df['train_acc'] - result_df['test_acc']
    result_df = result_df.sort_values('test_acc', ascending=False).reset_index(drop=True)
    return histories, result_df


def plot_validation_curves(histories: dict, title: str) -> None:
    epochs = np.arange(1, len(next(iter(histories.values()))['loss']) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    for name, history in histories.items():
        axes[0].plot(epochs, history['val_accuracy'], label=name, linewidth=2)
        axes[1].plot(epochs, history['val_loss'], label=name, linewidth=2)

    axes[0].set_title(f'{title} - Validation Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Validation Accuracy')
    axes[0].legend()

    axes[1].set_title(f'{title} - Validation Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Validation Loss')
    axes[1].legend()

    plt.show()


def show_result_table(result_df: pd.DataFrame, title: str) -> None:
    print(title)
    display(
        result_df[
            [
                'name',
                'use_batch_norm',
                'optimizer_name',
                'learning_rate',
                'batch_size',
                'train_acc',
                'val_acc',
                'test_acc',
                'best_val_acc',
                'best_epoch',
                'generalization_gap',
                'seconds',
            ]
        ].round(4)
    )


---
## Cell 10. 실험 A — Batch Normalization 사용/미사용 비교

여기서 확인할 것:

1. Validation accuracy가 더 빨리 올라가는가?
2. Loss 곡선이 더 안정적인가?
3. 최종 test accuracy가 더 좋은가?
4. 일반화 간극(train-test gap)이 줄어드는가?

**주의:**
작은 tabular 데이터에서는 BatchNorm이 항상 압도적으로 좋지는 않을 수 있습니다.
그래도 곡선이 얼마나 흔들리는지, 수렴 속도가 어떤지 비교하는 데는 충분히 유용합니다.


In [ ]:
batch_norm_configs = [
    {
        'name': 'No BatchNorm',
        'use_batch_norm': False,
        'optimizer_name': 'adam',
        'learning_rate': 0.001,
        'batch_size': 16,
        'epochs': 40,
    },
    {
        'name': 'With BatchNorm',
        'use_batch_norm': True,
        'optimizer_name': 'adam',
        'learning_rate': 0.001,
        'batch_size': 16,
        'epochs': 40,
    },
]

batch_norm_histories, batch_norm_results = run_experiments(batch_norm_configs)
show_result_table(batch_norm_results, '실험 A 결과')
plot_validation_curves(batch_norm_histories, 'Experiment A: BatchNorm')


---
## Cell 11. 실험 B — Optimizer 비교

이번에는 모델 차원은 그대로 두고 optimizer만 바꿉니다.

- `SGD`: 가장 기본적인 경사하강법
- `SGD + momentum`: 이전 기울기 방향을 일부 반영하여 더 빠르고 덜 흔들리게 이동
- `RMSprop`: 학습률을 파라미터별로 조절
- `Adam`: momentum + adaptive learning rate를 합친 실무 기본 선택지

**포인트:** optimizer는 텐서 shape을 바꾸지 않습니다.
바꾸는 것은 **가중치 업데이트 규칙**입니다.


In [ ]:
optimizer_configs = [
    {
        'name': 'SGD',
        'use_batch_norm': True,
        'optimizer_name': 'sgd',
        'learning_rate': 0.01,
        'momentum': 0.0,
        'batch_size': 16,
        'epochs': 40,
    },
    {
        'name': 'SGD + momentum',
        'use_batch_norm': True,
        'optimizer_name': 'sgd',
        'learning_rate': 0.01,
        'momentum': 0.9,
        'batch_size': 16,
        'epochs': 40,
    },
    {
        'name': 'RMSprop',
        'use_batch_norm': True,
        'optimizer_name': 'rmsprop',
        'learning_rate': 0.001,
        'batch_size': 16,
        'epochs': 40,
    },
    {
        'name': 'Adam',
        'use_batch_norm': True,
        'optimizer_name': 'adam',
        'learning_rate': 0.001,
        'batch_size': 16,
        'epochs': 40,
    },
]

optimizer_histories, optimizer_results = run_experiments(optimizer_configs)
show_result_table(optimizer_results, '실험 B 결과')
plot_validation_curves(optimizer_histories, 'Experiment B: Optimizers')


---
## Cell 12. 실험 C — Adam learning rate 비교

같은 Adam이라도 learning rate를 바꾸면 결과가 크게 달라집니다.

- 너무 크면: loss가 출렁이거나 발산할 수 있음
- 너무 작으면: 너무 천천히 학습함
- 적당하면: 빠르고 안정적으로 수렴

이번 실험은 **Adam이 무조건 좋다**가 아니라,
**Adam도 learning rate를 잘못 잡으면 성능이 떨어질 수 있다**는 점을 확인하는 데 목적이 있습니다.


In [ ]:
adam_lr_configs = [
    {
        'name': 'Adam lr=0.01',
        'use_batch_norm': True,
        'optimizer_name': 'adam',
        'learning_rate': 0.01,
        'batch_size': 16,
        'epochs': 40,
    },
    {
        'name': 'Adam lr=0.001',
        'use_batch_norm': True,
        'optimizer_name': 'adam',
        'learning_rate': 0.001,
        'batch_size': 16,
        'epochs': 40,
    },
    {
        'name': 'Adam lr=0.0003',
        'use_batch_norm': True,
        'optimizer_name': 'adam',
        'learning_rate': 0.0003,
        'batch_size': 16,
        'epochs': 40,
    },
    {
        'name': 'Adam lr=0.0001',
        'use_batch_norm': True,
        'optimizer_name': 'adam',
        'learning_rate': 0.0001,
        'batch_size': 16,
        'epochs': 40,
    },
]

adam_lr_histories, adam_lr_results = run_experiments(adam_lr_configs)
show_result_table(adam_lr_results, '실험 C 결과')
plot_validation_curves(adam_lr_histories, 'Experiment C: Adam Learning Rate')


---
## Cell 13. 실험 D — Batch size 비교

이번에는 batch size만 바꿉니다.

기억할 점:

- batch size는 **학습 단계에서 한 번에 넣는 샘플 수**입니다.
- 따라서 입력 shape은 매 step마다 `(batch_size, feature_count)`가 됩니다.
- 예를 들어 feature가 13개일 때
  - batch size 4  → `(4, 13)`
  - batch size 16 → `(16, 13)`
  - batch size 64 → `(64, 13)`

하지만 레이어 구조 자체는 변하지 않으므로,
은닉층/출력층의 두 번째 축은 그대로 유지됩니다.

예:

- 입력: `(16, 13)`
- hidden1: `(16, 64)`
- hidden2: `(16, 32)`
- output: `(16, 3)`


In [ ]:
batch_size_configs = [
    {
        'name': 'batch_size=4',
        'use_batch_norm': True,
        'optimizer_name': 'adam',
        'learning_rate': 0.001,
        'batch_size': 4,
        'epochs': 40,
    },
    {
        'name': 'batch_size=8',
        'use_batch_norm': True,
        'optimizer_name': 'adam',
        'learning_rate': 0.001,
        'batch_size': 8,
        'epochs': 40,
    },
    {
        'name': 'batch_size=16',
        'use_batch_norm': True,
        'optimizer_name': 'adam',
        'learning_rate': 0.001,
        'batch_size': 16,
        'epochs': 40,
    },
    {
        'name': 'batch_size=32',
        'use_batch_norm': True,
        'optimizer_name': 'adam',
        'learning_rate': 0.001,
        'batch_size': 32,
        'epochs': 40,
    },
]

batch_size_histories, batch_size_results = run_experiments(batch_size_configs)
show_result_table(batch_size_results, '실험 D 결과')
plot_validation_curves(batch_size_histories, 'Experiment D: Batch Size')


---
## Cell 14. 직접 바꿔보는 커스텀 실험 셀

아래 설정을 바꿔서 직접 실험해 보세요.

특히 다음 항목을 추천합니다.

1. `hidden_units=(32,)` 와 `hidden_units=(128, 64, 32)` 비교
2. `use_batch_norm=True/False` 비교
3. `optimizer_name='adam'` 에서 `learning_rate` 바꾸기
4. `batch_size`를 4, 16, 32로 바꾸기

**shape 관점에서 해석하기:**

- `hidden_units=(32,)`면 출력 흐름은 `(batch, 13) -> (batch, 32) -> (batch, 3)`
- `hidden_units=(128, 64, 32)`면 `(batch, 13) -> (batch, 128) -> (batch, 64) -> (batch, 32) -> (batch, 3)`
- 출력층 마지막 숫자는 항상 클래스 수 `3`이어야 합니다.


In [ ]:
my_config = {
    'name': 'My custom experiment',
    'use_batch_norm': True,
    'optimizer_name': 'adam',
    'learning_rate': 0.001,
    'batch_size': 16,
    'epochs': 40,
    'hidden_units': (128, 64, 32),
}

my_model, my_history, my_result = train_single_experiment(my_config)
display(pd.DataFrame([my_result]).round(4))
plot_validation_curves({'My custom experiment': my_history}, 'Custom Experiment')
my_model.summary()


---
## 핵심 개념 정리

### 1. 데이터 차원 읽기

- `X.shape = (샘플 수, 피처 수)`
- `y.shape = (샘플 수,)` 또는 원-핫 인코딩 후 `(샘플 수, 클래스 수)`

### 2. 배치 차원

- 학습 때 모델에 들어가는 입력은 보통 `(batch_size, 피처 수)`
- 배치 크기가 달라지면 **첫 번째 축만** 바뀜

### 3. 은닉층 차원

- `Dense(64)`면 출력 shape의 마지막 축이 64가 됨
- 따라서 `(batch_size, 13) -> (batch_size, 64)`

### 4. 출력층 차원

- 다중분류에서는 클래스 수와 같아야 함
- Wine 데이터셋은 클래스 3개이므로 최종 출력은 `(batch_size, 3)`

### 5. BatchNorm의 역할

- 출력 shape은 유지
- 내부 값의 분포를 안정화해 학습을 돕는 역할

### 6. Optimizer의 역할

- 출력 차원을 바꾸지 않음
- 손실을 줄이기 위해 가중치를 업데이트하는 방법을 바꿈

### 7. Adam learning rate의 의미

- 너무 크면 불안정
- 너무 작으면 너무 느림
- 적당한 값을 찾아야 함
